# Scraping Data Pemain Sepak Bola - Transfermarkt

Pipeline scraping untuk project klasifikasi kategori nilai pasar pemain sepak bola Big 5 Eropa.

Scope scraping:
- Liga: Premier League, La Liga, Bundesliga, Serie A, Ligue 1
- Musim: 2017 sampai 2024
- Level data: pemain per klub per musim
- Sumber: Transfermarkt

Output raw:
- `data/raw/clubs_raw.csv`
- `data/raw/players_raw.csv`
- `data/raw/scraping_log.csv`


## 1. Install dan Import Library


In [9]:

%pip install requests beautifulsoup4 pandas numpy lxml tqdm


In [10]:
from pathlib import Path
import os
import random
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.notebook import tqdm
from urllib3.util.retry import Retry

print('Library berhasil diimport.')


Library berhasil diimport.


## 2. Konfigurasi Project dan Folder


In [11]:
BASE_URL = 'https://www.transfermarkt.com'

FORCE_RESCRAPE = False
RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DATA_MODEL_DIR = PROJECT_ROOT / 'data' / 'model'
DATA_OUTPUT_DIR = PROJECT_ROOT / 'data' / 'output'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
APP_DIR = PROJECT_ROOT / 'app'
SRC_DIR = PROJECT_ROOT / 'src'

for folder in [
    DATA_RAW_DIR,
    DATA_PROCESSED_DIR,
    DATA_MODEL_DIR,
    DATA_OUTPUT_DIR,
    MODELS_DIR,
    REPORTS_FIGURES_DIR,
    APP_DIR,
    SRC_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

CLUBS_FILE = DATA_RAW_DIR / 'clubs_raw.csv'
PLAYERS_FILE = DATA_RAW_DIR / 'players_raw.csv'
SCRAPING_LOG_FILE = DATA_RAW_DIR / 'scraping_log.csv'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Referer': 'https://www.transfermarkt.com/',
}

LEAGUES = {
    'Premier League': {'url_name': 'premier-league', 'url_id': 'GB1', 'country': 'England', 'rank': 1},
    'La Liga': {'url_name': 'laliga', 'url_id': 'ES1', 'country': 'Spain', 'rank': 2},
    'Bundesliga': {'url_name': 'bundesliga', 'url_id': 'L1', 'country': 'Germany', 'rank': 3},
    'Serie A': {'url_name': 'serie-a', 'url_id': 'IT1', 'country': 'Italy', 'rank': 4},
    'Ligue 1': {'url_name': 'ligue-1', 'url_id': 'FR1', 'country': 'France', 'rank': 5},
}

SEASONS = [str(y) for y in range(2017, 2025)]

DELAY_MIN = 2.5
DELAY_MAX = 4.5
REQUEST_TIMEOUT = 20
MAX_RETRIES = 3
CHECKPOINT_EVERY_CLUBS = 25

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw output  : {DATA_RAW_DIR}')
print(f'Seasons     : {SEASONS}')
print(f'Force scrape: {FORCE_RESCRAPE}')


Project root: /content
Raw output  : /content/data/raw
Seasons     : ['2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Force scrape: False


## 3. Logging dan Request Session


In [12]:
scraping_logs = []
if SCRAPING_LOG_FILE.exists() and not FORCE_RESCRAPE:
    try:
        scraping_logs = pd.read_csv(SCRAPING_LOG_FILE).to_dict('records')
        print(f'Resume log scraping dari {len(scraping_logs)} baris.')
    except Exception as exc:
        scraping_logs = []
        print(f'Log lama tidak bisa dibaca, mulai log baru: {exc}')


def log_event(level, event, url='', league='', season='', club='', message='', extra=None):
    row = {
        'timestamp': datetime.utcnow().isoformat(timespec='seconds'),
        'level': level,
        'event': event,
        'url': url,
        'league': league,
        'season': season,
        'club': club,
        'message': message,
        'extra': extra if extra is not None else '',
    }
    scraping_logs.append(row)


def save_scraping_log():
    if scraping_logs:
        df_log = pd.DataFrame(scraping_logs)
    elif SCRAPING_LOG_FILE.exists():
        return
    else:
        df_log = pd.DataFrame(columns=[
            'timestamp', 'level', 'event', 'url', 'league', 'season', 'club', 'message', 'extra'
        ])
    df_log.to_csv(SCRAPING_LOG_FILE, index=False)


def build_session():
    session = requests.Session()
    retry = Retry(
        total=MAX_RETRIES,
        connect=MAX_RETRIES,
        read=MAX_RETRIES,
        backoff_factor=1.5,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=['GET'],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('https://', adapter)
    session.mount('http://', adapter)
    session.headers.update(HEADERS)
    return session


session = build_session()


def get_page(url, league='', season='', club=''):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
            response = session.get(url, timeout=REQUEST_TIMEOUT)
            status_code = response.status_code

            if status_code == 200:
                return BeautifulSoup(response.content, 'lxml')

            if status_code == 403:
                log_event('ERROR', 'http_403_forbidden', url, league, season, club, 'Access forbidden')
                print(f'HTTP 403 forbidden: {url}')
                return None

            if status_code == 404:
                log_event('WARNING', 'http_404_not_found', url, league, season, club, 'Page not found')
                print(f'HTTP 404 not found: {url}')
                return None

            if status_code == 429:
                wait_seconds = 45 + attempt * 20
                log_event('WARNING', 'http_429_rate_limit', url, league, season, club, f'Rate limited, wait {wait_seconds}s')
                print(f'HTTP 429 rate limited. Tunggu {wait_seconds}s...')
                time.sleep(wait_seconds)
                continue

            if 500 <= status_code < 600:
                log_event('WARNING', 'http_5xx_server_error', url, league, season, club, f'Server error {status_code}')
                print(f'HTTP {status_code} server error, attempt {attempt}/{MAX_RETRIES}: {url}')
                continue

            log_event('ERROR', 'http_unhandled_status', url, league, season, club, f'HTTP {status_code}')
            print(f'HTTP {status_code}: {url}')
            return None

        except requests.exceptions.Timeout as exc:
            log_event('WARNING', 'request_timeout', url, league, season, club, str(exc))
            print(f'Timeout attempt {attempt}/{MAX_RETRIES}: {url}')
        except requests.exceptions.RequestException as exc:
            log_event('WARNING', 'request_exception', url, league, season, club, str(exc))
            print(f'Request error attempt {attempt}/{MAX_RETRIES}: {exc}')

    log_event('ERROR', 'request_failed_after_retries', url, league, season, club, 'Max retries exceeded')
    return None


print('Session dan logging siap.')


Session dan logging siap.


## 4. Parser Market Value


In [14]:
def parse_market_value(value):
    if value is None:
        return np.nan

    text = str(value).strip()
    if text == '' or text == '-' or text.lower() in {'nan', 'none', 'n/a'}:
        return np.nan

    normalized = (
        text.replace('\xa0', ' ')
            .replace('\u202f', ' ')
            .replace('?', 'EUR')
            .strip()
    )
    normalized = re.sub(r'\s+', ' ', normalized)
    normalized_lower = normalized.lower()

    number_match = re.search(r'(\d+(?:[.,]\d+)?)', normalized_lower)
    if not number_match:
        return np.nan

    try:
        amount = float(number_match.group(1).replace(',', '.'))
    except ValueError:
        return np.nan

    unit_text = normalized_lower[number_match.end():].strip()

    if unit_text.startswith('bn') or re.search(r'billion|milyar', normalized_lower):
        return amount * 1000
    if unit_text.startswith('m') or re.search(r'million|mio', normalized_lower):
        return amount
    if unit_text.startswith('k') or re.search(r'thousand|ribu', normalized_lower):
        return amount / 1000

    return amount


parser_tests = {
    'EUR 500k': 0.5,
    'EUR 1.50m': 1.5,
    'EUR 75.00m': 75.0,
    'EUR 1.20bn': 1200.0,
    '-': np.nan,
    '?500k': 0.5,
    '? 1,50m': 1.5,
    ' EUR\xa075.00m ': 75.0,
}

for raw_value, expected in parser_tests.items():
    parsed = parse_market_value(raw_value)
    if pd.isna(expected):
        assert pd.isna(parsed), f'{raw_value}: expected NaN, got {parsed}'
    else:
        assert abs(parsed - expected) < 1e-9, f'{raw_value}: expected {expected}, got {parsed}'

print('Market value parser tests passed.')


Market value parser tests passed.


## 5. Helper Penyimpanan dan Validasi HTML


In [15]:
RAW_PLAYER_COLUMNS = [
    'player_id',
    'player_name',
    'player_url',
    'shirt_number',
    'pos_category',
    'position_detail',
    'age',
    'date_of_birth',
    'nationality',
    'height_m',
    'preferred_foot',
    'club',
    'club_total_mv_raw',
    'league',
    'league_rank',
    'season',
    'market_value_str',
    'market_value_mio',
]

RAW_CLUB_COLUMNS = [
    'league',
    'country',
    'league_rank',
    'season',
    'club_name',
    'club_url',
    'squad_size',
    'club_total_mv_raw',
]


def save_csv_atomic(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(temp_path, index=False)
    os.replace(temp_path, path)


def save_checkpoint(records, path, columns):
    df = pd.DataFrame(records)
    for column in columns:
        if column not in df.columns:
            df[column] = np.nan
    df = df[columns]
    save_csv_atomic(df, path)
    save_scraping_log()
    print(f'Checkpoint tersimpan: {path} ({len(df)} baris)')


def find_items_table(soup, url, league='', season='', club=''):
    if soup is None:
        return None

    table = soup.find('table', {'class': 'items'})
    if table is None:
        log_event('WARNING', 'html_table_missing', url, league, season, club, 'table.items not found')
        return None

    tbody = table.find('tbody')
    if tbody is None:
        log_event('WARNING', 'html_tbody_missing', url, league, season, club, 'tbody not found')
        return None

    return table


print('Helper penyimpanan dan validasi HTML siap.')


Helper penyimpanan dan validasi HTML siap.


## 6. Scraping Daftar Klub per Liga per Musim


In [16]:
def get_clubs(league_name, league_info, season):
    url = (
        f"{BASE_URL}/{league_info['url_name']}/startseite"
        f"/wettbewerb/{league_info['url_id']}/plus/?saison_id={season}"
    )
    soup = get_page(url, league=league_name, season=season)
    table = find_items_table(soup, url, league=league_name, season=season)
    if table is None:
        return []

    clubs = []
    rows = table.find('tbody').find_all('tr')
    if not rows:
        log_event('WARNING', 'club_rows_empty', url, league_name, season, '', 'No club rows found')
        return []

    for row_index, row in enumerate(rows):
        try:
            link_cell = row.find('td', {'class': 'hauptlink'})
            link = link_cell.find('a') if link_cell else None
            if not link or not link.get('href'):
                continue

            tds = row.find_all('td')
            club_total_mv_raw = tds[-1].text.strip() if tds else ''
            squad_size = tds[2].text.strip() if len(tds) > 2 else ''

            club_name = link.text.strip()
            club_url = link['href']
            if not club_name or not club_url:
                log_event('WARNING', 'club_row_invalid', url, league_name, season, club_name, 'Missing club name or URL')
                continue

            clubs.append({
                'league': league_name,
                'country': league_info['country'],
                'league_rank': league_info['rank'],
                'season': season,
                'club_name': club_name,
                'club_url': club_url,
                'squad_size': squad_size,
                'club_total_mv_raw': club_total_mv_raw,
            })
        except Exception as exc:
            log_event('WARNING', 'club_row_parse_failed', url, league_name, season, '', str(exc), extra=f'row_index={row_index}')
            continue

    return clubs


print('Fungsi get_clubs siap.')


Fungsi get_clubs siap.


In [17]:
if CLUBS_FILE.exists() and not FORCE_RESCRAPE:
    df_clubs = pd.read_csv(CLUBS_FILE)
    print(f'{CLUBS_FILE} sudah ada: {len(df_clubs)} baris. Scraping klub dilewati.')
else:
    all_clubs = []
    for league_name, league_info in LEAGUES.items():
        print(f'Liga: {league_name}')
        for season in tqdm(SEASONS, desc=league_name):
            clubs = get_clubs(league_name, league_info, season)
            all_clubs.extend(clubs)
            if len(all_clubs) > 0 and len(all_clubs) % CHECKPOINT_EVERY_CLUBS == 0:
                save_checkpoint(all_clubs, CLUBS_FILE, RAW_CLUB_COLUMNS)

    df_clubs = pd.DataFrame(all_clubs).drop_duplicates()
    for column in RAW_CLUB_COLUMNS:
        if column not in df_clubs.columns:
            df_clubs[column] = np.nan
    df_clubs = df_clubs[RAW_CLUB_COLUMNS]
    save_csv_atomic(df_clubs, CLUBS_FILE)
    save_scraping_log()
    print(f'Total klub: {len(df_clubs)} baris')

df_clubs.head()


Liga: Premier League


Premier League:   0%|          | 0/8 [00:00<?, ?it/s]

Checkpoint tersimpan: /content/data/raw/clubs_raw.csv (100 baris)


Liga: La Liga


La Liga:   0%|          | 0/8 [00:00<?, ?it/s]

...

Serie A:   0%|          | 0/8 [00:00<?, ?it/s]

Liga: Ligue 1


Ligue 1:   0%|          | 0/8 [00:00<?, ?it/s]

Total klub: 780 baris


,league,country,league_rank,season,club_name,club_url,squad_size,club_total_mv_raw
0,Premier League,England,1,2017,Manchester City,/manchester-city/startseite/verein/281/saison_...,32,€1.01bn
1,Premier League,England,1,2017,Chelsea FC,/fc-chelsea/startseite/verein/631/saison_id/2017,46,€884.25m
2,Premier League,England,1,2017,Liverpool FC,/fc-liverpool/startseite/verein/31/saison_id/2017,35,€857.50m
3,Premier League,England,1,2017,Manchester United,/manchester-united/startseite/verein/985/saiso...,35,€849.50m
4,Premier League,England,1,2017,Tottenham Hotspur,/tottenham-hotspur/startseite/verein/148/saiso...,34,€829.60m


## 7. Scraping Data Pemain per Klub


In [18]:
def extract_player_id(player_url):
    if not player_url:
        return None
    match = re.search(r'/spieler/(\d+)', player_url)
    if match:
        return match.group(1)
    match = re.search(r'/(\d+)$', player_url)
    return match.group(1) if match else None


def get_player_cell_by_index(tds, index):
    return tds[index] if len(tds) > index else None


def get_players(club_url, club_name, club_total_mv_raw, league, league_rank, season):
    try:
        parts = str(club_url).split('/')
        club_slug = parts[1]
        club_id = parts[4] if len(parts) > 4 else parts[-1]
    except Exception as exc:
        log_event('WARNING', 'club_url_parse_failed', '', league, season, club_name, str(exc), extra=str(club_url))
        return []

    url = f'{BASE_URL}/{club_slug}/kader/verein/{club_id}/saison_id/{season}/plus/1'
    soup = get_page(url, league=league, season=season, club=club_name)
    table = find_items_table(soup, url, league=league, season=season, club=club_name)
    if table is None:
        return []

    rows = table.find('tbody').find_all('tr', {'class': ['odd', 'even']})
    if not rows:
        log_event('WARNING', 'player_rows_empty', url, league, season, club_name, 'No player rows found')
        return []

    players = []

    for row_index, row in enumerate(rows):
        try:
            tds = row.find_all('td')
            if len(tds) < 10:
                log_event('WARNING', 'player_row_too_short', url, league, season, club_name, 'Not enough td cells', extra=f'row_index={row_index};td_count={len(tds)}')
                continue

            shirt_td = get_player_cell_by_index(tds, 0)
            shirt_div = shirt_td.find('div', {'class': 'rn_nummer'}) if shirt_td else None
            shirt_number = shirt_div.text.strip() if shirt_div else ''
            pos_category = shirt_td.get('title', '').strip() if shirt_td else ''

            player_link = None
            for cell in tds:
                candidate = cell.find('a', href=re.compile(r'/spieler/\d+'))
                if candidate:
                    player_link = candidate
                    break

            if not player_link:
                log_event('WARNING', 'player_link_missing', url, league, season, club_name, 'Player link not found', extra=f'row_index={row_index}')
                continue

            player_name = player_link.text.strip()
            player_url = player_link.get('href', '').strip()
            player_id = extract_player_id(player_url)

            if not player_name:
                log_event('WARNING', 'player_name_missing', url, league, season, club_name, 'Player name is empty', extra=f'row_index={row_index}')
                continue

            position_detail = ''
            name_cell_index = next((idx for idx, cell in enumerate(tds) if cell.find('a', href=re.compile(r'/spieler/\d+'))), None)
            if name_cell_index is not None:
                position_cell = get_player_cell_by_index(tds, name_cell_index + 1)
                position_detail = position_cell.text.strip() if position_cell else ''

            dob = ''
            age = ''
            dob_text = ''
            for cell in tds:
                text = cell.text.strip()
                if re.search(r'\(\d{1,2}\)', text) and re.search(r'\d{1,2}/\d{1,2}/\d{4}', text):
                    dob_text = text
                    break
            if dob_text:
                age_match = re.search(r'\((\d+)\)', dob_text)
                dob_match = re.search(r'\d{1,2}/\d{1,2}/\d{4}', dob_text)
                age = age_match.group(1) if age_match else ''
                dob = dob_match.group(0) if dob_match else ''

            nat_imgs = row.find_all('img', {'class': 'flaggenrahmen'})
            nationality = nat_imgs[0].get('title', '').strip() if nat_imgs else ''

            height_m = np.nan
            for cell in tds:
                height_text = cell.text.strip()
                height_match = re.search(r'\b(1[.,]\d{2}|2[.,][0-2]\d?)m?\b', height_text)
                if height_match:
                    try:
                        candidate_height = float(height_match.group(1).replace(',', '.'))
                        if 1.4 <= candidate_height <= 2.2:
                            height_m = candidate_height
                            break
                    except ValueError:
                        pass

            preferred_foot = ''
            for cell in tds:
                foot_text = cell.text.strip().lower()
                if foot_text in {'left', 'right', 'both'}:
                    preferred_foot = foot_text
                    break

            mv_str = tds[-1].text.strip() if tds else ''
            mv_mio = parse_market_value(mv_str)

            player_record = {
                'player_id': player_id,
                'player_name': player_name,
                'player_url': player_url,
                'shirt_number': shirt_number,
                'pos_category': pos_category,
                'position_detail': position_detail,
                'age': age,
                'date_of_birth': dob,
                'nationality': nationality,
                'height_m': height_m,
                'preferred_foot': preferred_foot,
                'club': club_name,
                'club_total_mv_raw': club_total_mv_raw,
                'league': league,
                'league_rank': league_rank,
                'season': season,
                'market_value_str': mv_str,
                'market_value_mio': mv_mio,
            }

            required_values = {
                'player_name': player_record['player_name'],
                'season': player_record['season'],
                'league': player_record['league'],
                'club': player_record['club'],
            }
            missing_required = [key for key, value in required_values.items() if pd.isna(value) or str(value).strip() == '']
            if missing_required:
                log_event('WARNING', 'player_row_validation_failed', url, league, season, club_name, 'Missing required values', extra=','.join(missing_required))
                continue

            if not player_id:
                log_event('INFO', 'player_id_missing', url, league, season, club_name, 'Player ID could not be parsed', extra=player_url)

            players.append(player_record)
        except Exception as exc:
            log_event('WARNING', 'player_row_parse_failed', url, league, season, club_name, str(exc), extra=f'row_index={row_index}')
            continue

    return players


print('Fungsi get_players siap.')


Fungsi get_players siap.


In [19]:
# Verifikasi satu klub sebelum scrape massal. Cell ini aman dilewati jika ingin langsung scrape massal.
test_players = get_players(
    club_url='/manchester-city/startseite/verein/281',
    club_name='Manchester City',
    club_total_mv_raw='',
    league='Premier League',
    league_rank=1,
    season='2024',
)

print(f'Jumlah pemain test: {len(test_players)}')
if test_players:
    sample = test_players[0]
    preview_columns = [
        'player_id', 'player_name', 'position_detail', 'pos_category', 'age',
        'date_of_birth', 'height_m', 'preferred_foot', 'market_value_str', 'market_value_mio'
    ]
    print({column: sample.get(column) for column in preview_columns})

save_scraping_log()


Jumlah pemain test: 44
{'player_id': '238223', 'player_name': 'Ederson', 'position_detail': '', 'pos_category': 'Goalkeeper', 'age': '31', 'date_of_birth': '17/08/1993', 'height_m': 1.88, 'preferred_foot': 'left', 'market_value_str': '€20.00m', 'market_value_mio': 20.0}


In [20]:
if not CLUBS_FILE.exists():
    raise FileNotFoundError(f'{CLUBS_FILE} belum ada. Jalankan scraping klub terlebih dahulu.')

df_clubs = pd.read_csv(CLUBS_FILE)

for column in RAW_CLUB_COLUMNS:
    if column not in df_clubs.columns:
        raise ValueError(f'Kolom wajib tidak ada di clubs_raw.csv: {column}')

if PLAYERS_FILE.exists() and not FORCE_RESCRAPE:
    df_done = pd.read_csv(PLAYERS_FILE)
    all_players = df_done.to_dict('records')
    done_keys = set(zip(
        df_done['league'].astype(str),
        df_done['club'].astype(str),
        df_done['season'].astype(str),
    ))
    print(f'Resume dari {len(all_players)} baris di {PLAYERS_FILE}.')
else:
    all_players = []
    done_keys = set()
    if PLAYERS_FILE.exists() and FORCE_RESCRAPE:
        print('FORCE_RESCRAPE=True, scraping pemain dimulai ulang dan file akan ditimpa setelah checkpoint baru dibuat.')
    else:
        print('Mulai scraping pemain dari awal.')

for _, row in tqdm(df_clubs.iterrows(), total=len(df_clubs), desc='Scraping pemain'):
    key = (str(row['league']), str(row['club_name']), str(row['season']))
    if key in done_keys:
        continue

    players = get_players(
        club_url=row['club_url'],
        club_name=row['club_name'],
        club_total_mv_raw=row.get('club_total_mv_raw', ''),
        league=row['league'],
        league_rank=row['league_rank'],
        season=row['season'],
    )
    all_players.extend(players)
    done_keys.add(key)

    if len(all_players) > 0 and len(all_players) % 500 < max(len(players), 1):
        save_checkpoint(all_players, PLAYERS_FILE, RAW_PLAYER_COLUMNS)

save_checkpoint(all_players, PLAYERS_FILE, RAW_PLAYER_COLUMNS)

df_players = pd.read_csv(PLAYERS_FILE)
print(f'Total pemain raw: {len(df_players)} baris')
df_players.head()


Mulai scraping pemain dari awal.


Scraping pemain:   0%|          | 0/780 [00:00<?, ?it/s]

Checkpoint tersimpan: /content/data/raw/players_raw.csv (530 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (1030 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (1508 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (2019 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (2523 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (3009 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (3509 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (4008 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (4513 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (5038 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (5538 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (6019 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (6516 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (7020 baris)


Checkpoint tersimpan: /content/data/raw/players_raw.csv (7500 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (8020 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (8511 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (9019 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (9506 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (10036 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (10518 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (11013 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (11529 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (12016 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (12500 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (13003 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (13505 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (14004 baris)
Checkpoint tersimpan: /co

/tmp/ipykernel_12766/3154464251.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp': datetime.utcnow().isoformat(timespec='seconds'),


...

/tmp/ipykernel_12766/3154464251.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp': datetime.utcnow().isoformat(timespec='seconds'),


HTTP 502 server error, attempt 1/3: https://www.transfermarkt.com/fc-bayern-munchen/kader/verein/27/saison_id/2024/plus/1


Checkpoint tersimpan: /content/data/raw/players_raw.csv (17037 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (17528 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (18039 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (18519 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (19034 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (19504 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (20012 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (20512 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (21011 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (21503 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (22032 baris)


Checkpoint tersimpan: /content/data/raw/players_raw.csv (22524 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (23010 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (23534 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (24004 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (24512 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (25021 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (25520 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (26010 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (26518 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (27021 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (27509 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (28012 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (28510 baris)
Checkpoint tersimpan: /content/data/raw/players_raw.csv (29019 baris)
Checkpoint tersimpan

,player_id,player_name,player_url,shirt_number,pos_category,position_detail,age,date_of_birth,nationality,height_m,preferred_foot,club,club_total_mv_raw,league,league_rank,season,market_value_str,market_value_mio
0,238223,Ederson,/ederson/profil/spieler/238223,31,Goalkeeper,NaN,24.0,17/08/1993,Brazil,1.88,left,Manchester City,€1.01bn,Premier League,1,2017,€50.00m,50.0
1,40204,Joe Hart,/joe-hart/profil/spieler/40204,-,Goalkeeper,NaN,31.0,19/04/1987,England,1.96,right,Manchester City,€1.01bn,Premier League,1,2017,€10.00m,10.0
2,40423,Claudio Bravo,/claudio-bravo/profil/spieler/40423,1,Goalkeeper,NaN,35.0,13/04/1983,Chile,1.84,right,Manchester City,€1.01bn,Premier League,1,2017,€3.50m,3.5
3,201574,Angus Gunn,/angus-gunn/profil/spieler/201574,54,Goalkeeper,NaN,22.0,22/01/1996,Scotland,1.96,right,Manchester City,€1.01bn,Premier League,1,2017,€2.00m,2.0
4,186590,John Stones,/john-stones/profil/spieler/186590,5,Defender,NaN,24.0,28/05/1994,England,1.88,right,Manchester City,€1.01bn,Premier League,1,2017,€50.00m,50.0


## 8. Validasi Ringkas Raw Output


In [21]:
if not PLAYERS_FILE.exists():
    raise FileNotFoundError(f'{PLAYERS_FILE} belum ada. Jalankan scraping pemain terlebih dahulu.')

df_players = pd.read_csv(PLAYERS_FILE)

missing_columns = [column for column in RAW_PLAYER_COLUMNS if column not in df_players.columns]
if missing_columns:
    raise ValueError(f'Kolom raw wajib belum tersedia: {missing_columns}')

print(f'File players : {PLAYERS_FILE}')
print(f'Total baris  : {len(df_players)}')
print(f'Kolom        : {list(df_players.columns)}')
print(f'Musim        : {sorted(df_players["season"].dropna().astype(str).unique())}')
print(f'Liga         : {df_players["league"].value_counts(dropna=False).to_dict()}')
print()
print('Null per kolom:')
print(df_players.isna().sum().sort_values(ascending=False))

save_scraping_log()
print(f'Log scraping : {SCRAPING_LOG_FILE}')


File players : /content/data/raw/players_raw.csv
Total baris  : 30024
Kolom        : ['player_id', 'player_name', 'player_url', 'shirt_number', 'pos_category', 'position_detail', 'age', 'date_of_birth', 'nationality', 'height_m', 'preferred_foot', 'club', 'club_total_mv_raw', 'league', 'league_rank', 'season', 'market_value_str', 'market_value_mio']
Musim        : ['2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Liga         : {'Serie A': 6935, 'Premier League': 6386, 'La Liga': 5886, 'Ligue 1': 5759, 'Bundesliga': 5058}

Null per kolom:
position_detail      30024
market_value_mio      1690
preferred_foot         479
height_m               410
age                     10
date_of_birth           10
player_name              0
player_id                0
pos_category             0
shirt_number             0
nationality              0
player_url               0
club_total_mv_raw        0
club                     0
league                   0
league_rank              0
season  

## 9. Output Task 1

Notebook scraping ini hanya membuat raw output:
- `data/raw/clubs_raw.csv`
- `data/raw/players_raw.csv`
- `data/raw/scraping_log.csv`

Preprocessing, training model, dan dashboard belum dibuat pada task ini.
